# Heavy: SiamMiTB3Diff

One shared MiT-B3 encoder (3, 4, 18, 3 blocks), four absolute temporal differences,
and a compact convolutional decoder. This project model is not official ChangeFormerV6.

The primary experiment resizes each **whole 1024×1024 scene to 256×256**. Native resolution
retains the whole 1024×1024 scene. Neither regime uses crops or tiles.
Safe Run All uses CPU synthetic inputs only. All data/training/profiling paths are opt-in.

In [ ]:
import json
from copy import deepcopy
from pathlib import Path
from uuid import uuid4

import torch

from satchange.data import create_dataset_dataloader, prepare_levir_cd
from satchange.execution import seed_everything
from satchange.losses import BCEDiceLoss
from satchange.models import count_model_parameters, create_change_model
from satchange.optimization import run_parameter_search
from satchange.run import (
    probe_training_step,
    reconstruct_checkpoint,
    resolve_parameters,
    run_training,
)
from satchange.training import evaluate_thresholds
from satchange.utils import load_config

current = Path.cwd().resolve()
project_root = current.parent if current.name == "notebooks" else current
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError("Run from the repository root or notebooks directory.")
REGIME = "resized"  # Or "full_resolution"; safe synthetic smoke remains small.
config_path = project_root / "configs" / f"siam_mit_b3_diff_{REGIME}.yaml"
config = load_config(config_path)
print(config_path, config["data"], config["runtime"])

In [ ]:
RUN_DATA_PREPARATION = False
RUN_LOADERS = False
RUN_MEMORY_PROBE = False
RUN_FULL_RESOLUTION_PROBE = False
RUN_TINY_OVERFIT = False
RUN_TRAINING = False
RUN_HPO = False
RUN_CHECKPOINT_INSPECTION = False
RUN_FINAL_TEST = False
TRAIN_DEVICE = "cuda:0"  # Explicitly select an allocated idle device before enabling work.
HPO_GPU_IDS = [0]  # Logical visible-device indices; extend only for allocated GPUs.
CHECKPOINT_PATH = None  # Trusted project checkpoint, supplied explicitly.
THRESHOLDS = config["training"]["thresholds"]
dataset_root = project_root / config["dataset"]["prepared_directory"]

## Default architecture and safe synthetic forward

The default has 45,772,961 parameters, with 44,072,128 shared encoder parameters.
For 256×256: encoder maps are 64×64×64, 128×32×32, 320×16×16, 512×8×8.
Attention has 4096/1024/256/64 queries and 64 keys per stage, each head width 64.
The smoke below uses the **complete default depth**, at 64×64 on CPU.

In [ ]:
seed_everything(config["project"]["seed"])
previous_threads = torch.get_num_threads()
torch.set_num_threads(min(2, previous_threads))
try:
    smoke_model = create_change_model(config["model"]["name"], config["model"]["parameters"]).eval()
    print(count_model_parameters(smoke_model))
    with torch.no_grad():
        logits = smoke_model(torch.rand(1, 3, 64, 64), torch.rand(1, 3, 64, 64))
    assert logits.shape == (1, 1, 64, 64) and logits.isfinite().all()
    print(
        "CPU default-depth logits:", tuple(logits.shape), "finite:", bool(logits.isfinite().all())
    )
finally:
    torch.set_num_threads(previous_threads)
del smoke_model, logits

## Optional preparation and loader inspection
Preparation is separate from training; training requires an existing dataset.

In [ ]:
if RUN_DATA_PREPARATION:
    dataset_root = prepare_levir_cd(config, project_root)
if RUN_LOADERS:
    _, inspection_loader = create_dataset_dataloader(
        dataset_root,
        "train",
        config["data"]["image_size"],
        config["data"]["batch_size"],
        config["data"]["num_workers"],
    )
    print("Training batches:", len(inspection_loader))
    del inspection_loader

## Optional bounded real training-memory probe

One fresh model/AdamW step, including optimizer-state allocation. No automatic batch search.
Native batch 1 additionally requires RUN_FULL_RESOLUTION_PROBE. Results distinguish peak
allocated from peak reserved MiB. FP32 is default; set runtime.precision="bf16" explicitly
for a supported CUDA device. No GPU measurements were made in the CPU implementation environment.

In [ ]:
if RUN_MEMORY_PROBE:
    result = probe_training_step(
        deepcopy(config),
        torch.device(TRAIN_DEVICE),
        enabled=True,
        allow_native=RUN_FULL_RESOLUTION_PROBE,
    )
    destination = project_root / "artifacts" / "profiling" / f"{uuid4().hex}.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(result)

## Optional tiny overfit

Four training samples serve as both fitting and validation data intentionally. This is a
pipeline diagnostic, not a research validation score. Each call creates fresh loaders,
model, loss, optimizer, scheduler and a unique directory, seeding before construction.

In [ ]:
if RUN_TINY_OVERFIT:
    tiny_config = deepcopy(config)
    tiny_config["training"]["epochs"] = 20
    tiny_history = run_training(
        tiny_config, project_root, torch.device(TRAIN_DEVICE), tiny_samples=4
    )
    print(
        [
            {
                key: row[key]
                for key in (
                    "epoch",
                    "train_loss",
                    "validation_loss",
                    "validation_f1",
                    "validation_iou",
                    "best_threshold",
                )
            }
            for row in tiny_history
        ]
    )

## Optional resized/native training

The provisional recipe is AdamW 1e-4, weight decay 1e-2, BCE/Dice 0.3/0.7, pos_weight 10,
seed 42, 100 future epochs, batch size one. It is not an HPO winner; pos_weight=10 is a recorded
provisional constant, not a freshly measured training-set estimate. Checkpoint selection uses
maximum validation F1 across the existing 0.05–0.95 grid, choosing the first threshold on ties.

In [ ]:
if RUN_TRAINING:
    history = run_training(deepcopy(config), project_root, torch.device(TRAIN_DEVICE))
    print(history[-1])

## Optional independent HPO

One independent trial per selected GPU; this is not DDP. Complete depth sequences are one
candidate each. Non-default depths are labeled depth ablations and retain their actual counts.

In [ ]:
SEARCH_SPACE = json.loads((project_root / "configs" / "siam_mit_b3_diff_search.json").read_text())
if RUN_HPO:
    results = run_parameter_search(
        search_space=deepcopy(SEARCH_SPACE),
        fixed_parameters=resolve_parameters(config),
        model_name=config["model"]["name"],
        dataset_root=dataset_root,
        thresholds=THRESHOLDS,
        number_of_epochs=config["training"]["epochs"],
        output_root=project_root / "artifacts" / "hyperparameter_search",
        search_name=f"heavy_{REGIME}_{uuid4().hex[:12]}",
        gpu_ids=HPO_GPU_IDS,
    )
    print(results)

## Trusted checkpoint reconstruction and final evaluation

The model reconstructs from saved architecture metadata with strict state loading. Resume also
restores optimizer, scheduler, history and per-rank RNG state; use the original run name,
precision and world size through the terminal --resume option. A fresh initialized experiment
is distinct from resume. Final test evaluation uses **only the threshold selected on validation**.

In [ ]:
if RUN_CHECKPOINT_INSPECTION or RUN_FINAL_TEST:
    if CHECKPOINT_PATH is None:
        raise ValueError("Set CHECKPOINT_PATH to a trusted project checkpoint.")
    restored_model, checkpoint = reconstruct_checkpoint(CHECKPOINT_PATH, torch.device(TRAIN_DEVICE))
    print(checkpoint["epoch"], checkpoint["best_metrics"])
    if RUN_FINAL_TEST:
        saved = checkpoint["config"]
        parameters = resolve_parameters(saved) if "model" in saved else saved
        _, final_loader = create_dataset_dataloader(
            dataset_root,
            "test",
            parameters["image_size"],
            parameters["batch_size"],
            parameters["num_workers"],
        )
        final_loss = BCEDiceLoss(
            bce_weight=parameters["bce_weight"],
            dice_weight=parameters["dice_weight"],
            pos_weight=parameters["positive_weight"],
        )
        final_loss = final_loss.to(torch.device(TRAIN_DEVICE))
        final_result = evaluate_thresholds(
            restored_model,
            final_loader,
            final_loss,
            torch.device(TRAIN_DEVICE),
            [checkpoint["current_metrics"]["best_threshold"]],
            precision=checkpoint.get("precision", "fp32"),
        )
        print(final_result)
    del restored_model, checkpoint

## Terminal DDP (not executed)

Run from the repository root, with the project environment active and GPUs already allocated:

```bash
python -m satchange.run train --config configs/siam_mit_b3_diff_resized.yaml --device cuda:0
python -m satchange.run train --config configs/siam_mit_b3_diff_full_resolution.yaml --device cuda:0 --precision bf16
python -m torch.distributed.run --standalone --nproc_per_node=4 -m satchange.run train --config configs/siam_mit_b3_diff_resized.yaml --device cuda --precision bf16 --ddp
python -m satchange.run hpo --config configs/siam_mit_b3_diff_resized.yaml --gpu-ids 0 1 2 3 --search-space configs/siam_mit_b3_diff_search.json --precision bf16
```

DDP replicates model state, and data.batch_size counts pairs **per process**; global batch is
batch size × world size. It does not combine four GPUs into one memory address space.
Training DistributedSampler pads to equal rank lengths; validation runs in full once on rank 0,
using the unwrapped module. Results and failures are broadcast; only rank 0 writes checkpoints.
Legacy BatchNorm remains local, without SyncBatchNorm conversion. Dice is computed per sample
and then averaged by the existing loss; gradient averaging preserves rank-local loss semantics.
No accumulation, FSDP, automatic precision conversion, or HPO-inside-DDP is enabled.